# 02 - Train ResNet-50 Ensemble (3-slice v2)

Entraîne 5 ResNet-50 ImageNet pretrained sur le dataset 3-slice v2 (crop_margin=30). Checkpoints sauvés sous le préfixe `resnet50_3slice_fold_*.pth`. ~120 min sur T4 (3x plus lent que ResNet-18 v1).

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'clean-structure',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run — data/ et work_dirs/ doivent déjà être peuplés.')

→ detect GPU
✓ deps already installed, skipping
✓  /content/INF8225_Projet/data already linked
✓  /content/INF8225_Projet/work_dirs already linked
✓  /content/INF8225_Projet/MedSAM/work_dir already linked
✓  /content/INF8225_Projet/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth already linked

Project : /content/INF8225_Projet
Drive   : /content/drive/MyDrive/Projet_Medsam
Device  : NVIDIA A100-SXM4-40GB
Torch   : 2.4.0+cu121

⚠  numpy was already loaded in this kernel before setup reinstalled it.
   Runtime → Restart session, then run your imports again
   (no need to rerun setup — deps are pinned on disk).
cwd: /content/INF8225_Projet


In [ ]:
#@title Verify 3-slice v2 dataset is present
from pathlib import Path

required = [
    Path("outputs/msd_implementation/resnet50_wide_crop/datasets/classifier_dataset_resnet50_wide_crop/train/0"),
    Path("outputs/msd_implementation/resnet50_wide_crop/datasets/classifier_dataset_resnet50_wide_crop/train/1"),
    Path("outputs/msd_implementation/resnet50_wide_crop/datasets/classifier_dataset_resnet50_wide_crop/val/0"),
    Path("outputs/msd_implementation/resnet50_wide_crop/datasets/classifier_dataset_resnet50_wide_crop/val/1"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing: {missing}. Lance d'abord le notebook 01 v2."

OK      data/classifier_dataset_3slice_v2/train/0
OK      data/classifier_dataset_3slice_v2/train/1
OK      data/classifier_dataset_3slice_v2/val/0
OK      data/classifier_dataset_3slice_v2/val/1


In [ ]:
#@title Run pipeline step (ResNet-50 training)
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet50_wide_crop.train_classifier"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'experiments.three_slice_v2.train_v2'], returncode=0)

In [ ]:
#@title Inspect ResNet-50 3-slice artifacts
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
print("checkpoint_dir =", checkpoint_dir)
for i in range(1, 6):
    ckpt = checkpoint_dir / f"resnet50_wide_crop_fold_{i}.pth"
    thr = checkpoint_dir / f"threshold_resnet50_wide_crop_run_{i}.txt"
    print(f"fold {i}: ckpt={ckpt.exists()} threshold={thr.exists()}")
assert all((checkpoint_dir / f"resnet50_wide_crop_fold_{i}.pth").exists() for i in range(1, 6)), "Missing 3-slice v2 ResNet-50 checkpoint(s)"

checkpoint_dir = /content/drive/MyDrive/Projet_Medsam
fold 1: ckpt=True threshold=True
fold 2: ckpt=True threshold=True
fold 3: ckpt=True threshold=True
fold 4: ckpt=True threshold=True
fold 5: ckpt=True threshold=True
